# generator-loss-fool-discriminator — faded example 3: Fill the discriminator's fake-batch target

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `generator-loss-fool-discriminator`. The last cell reports your progress on the `GAN: Generator loss to fool D` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `GAN: Generator loss to fool D` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`generator-loss-fool-discriminator`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "generator-loss-fool-discriminator"
DD_SUBTOPIC = "GAN: Generator loss to fool D"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

On the same fake batch, the discriminator's loss uses a target of 0 ('fake means 0') while the generator's uses 1. The flipped label is the entire adversarial signal.

## Faded exercise 3

Implement `both_losses(d_pred_fake)`. Return `(d_loss, g_loss)` where `d_loss` is BCE against an all-zeros target and `g_loss` is BCE against an all-ones target. Complete the blanked discriminator target.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import torch as t
import torch.nn.functional as F

t.manual_seed(5)

def both_losses(d_pred_fake):
    d_target = None  # TODO: fill in this step — read the prompt cell above
    d_loss = F.binary_cross_entropy(d_pred_fake, d_target)
    g_loss = F.binary_cross_entropy(d_pred_fake, t.ones_like(d_pred_fake))
    return d_loss, g_loss

print(both_losses(t.full((4,), 0.3)))


def _test():
    pred = t.tensor([0.1, 0.4, 0.6, 0.9])
    d_loss, g_loss = both_losses(pred)
    # independent ground truth: BCE against 0 is -log(1-p); against 1 is -log(p)
    d_expected = (-t.log(1 - pred)).mean()
    g_expected = (-t.log(pred)).mean()
    assert t.allclose(d_loss, d_expected, atol=1e-5), (d_loss, d_expected)
    assert t.allclose(g_loss, g_expected, atol=1e-5), (g_loss, g_expected)
    # opposite direction on a fake-confident batch
    d2, g2 = both_losses(t.full((4,), 0.05))
    assert float(d2) < float(g2)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn.functional as F

t.manual_seed(5)

def both_losses(d_pred_fake):
    d_target = t.zeros_like(d_pred_fake)
    d_loss = F.binary_cross_entropy(d_pred_fake, d_target)
    g_loss = F.binary_cross_entropy(d_pred_fake, t.ones_like(d_pred_fake))
    return d_loss, g_loss

print(both_losses(t.full((4,), 0.3)))
```
</details>